In [1]:
import pandas as pd 
import numpy as np 
import statsmodels.api as sm 

In [9]:
num_simulations = 10000
significant = 0

for i in range(num_simulations):
    
    num = 1000 
    
    event_time = int(num / 2) 
    
    R_market = np.random.normal(0, 1, num) + np.arange(num) / num 
    
    R_target = 2 + R_market + np.random.normal(0, 1, num) + (np.arange(num) == int(num / 2) + 1) * 2 
    
    results = sm.OLS(R_target[:event_time], sm.add_constant(R_market[:event_time])).fit() 
    
    alpha, beta = results.params 
    
    resid = R_target - results.predict(sm.add_constant(R_market)) 
    
    t_stat = resid[event_time + 1] / resid[:event_time].std(ddof = 2)

    if abs(t_stat) > 1.96:
        significant += 1

print(significant/num_simulations) 

0.5306


In [11]:
num = 1000

detected = 0
total = 0

for event_time in range(2, num - 1):

    np.random.seed(0)
    
    R_market = np.random.normal(0, 1, num) + np.arange(num)/num

    R_target = (2 + R_market + np.random.normal(0, 1, num) + (np.arange(num) == int(num/2) + 1) * 2)

    results = sm.OLS(R_target[:event_time], sm.add_constant(R_market[:event_time])).fit()

    resid = R_target - results.predict(sm.add_constant(R_market))

    t_stat = resid[event_time + 1]/resid[:event_time].std(ddof = 2)

    if abs(t_stat) > 1.96:
        detected += 1
    
    total += 1

print(detected/total)

/usr/local/python/3.14.2/lib/python3.14/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/usr/local/python/3.14.2/lib/python3.14/site-packages/numpy/_core/_methods.py:211: RuntimeWarning: divide by zero encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


0.04714142427281846


In [18]:
num_simulations = 1000

fractions = []

for seed in range(num_simulations):

    np.random.seed(seed)

    num = 1000
    event_time = 500

    R_market = np.random.normal(0, 1, num) + np.arange(num)/num

    R_target = 2 + R_market + np.random.normal(0, 1, num) + (np.arange(num) == 501) * 2

    results = sm.OLS(R_target[:event_time], sm.add_constant(R_market[:event_time])).fit()

    resid = R_target - results.predict(sm.add_constant(R_market))

    actual_t = resid[501] /resid[:500].std(ddof = 2)

    placebo_t = []

    placebo_times = list(range(480, 500)) + list(range(502, 522))

    for t in placebo_times: 

        results = sm.OLS(R_target[:t], sm.add_constant(R_market[:t])).fit()

        resid = R_target - results.predict(sm.add_constant(R_market))

        t_stat = resid[t + 1]/resid[:t].std(ddof = 2)

        placebo_t.append(t_stat)

    fractions.append(sum(x > actual_t for x in placebo_t)/40)

print(np.mean(fractions))

0.072975


In [15]:
def make_error(corr_const, num): 
 
 
  sigma = 5 * 1 / np.sqrt((1 - corr_const)**2 / (1 - corr_const**2)) 
 
 
  err = list() 
 
 
  prev = np.random.normal(0, sigma) 
 
 
  for n in range(num): 
 
 
    prev = corr_const * prev + (1 - corr_const) * np.random.normal(0, sigma) 
 
 
    err.append(prev) 
 
 
  return np.array(err) 

num = 1000

detected = 0
total = 0

for event_time in range(2, num - 1):

    np.random.seed(0)
    
    R_market = np.random.normal(0, 1, num) + np.arange(num)/num

    R_target = (2 + R_market + make_error(0.9, 1000) + (np.arange(num) == int(num/2) + 1) * 2)

    results = sm.OLS(R_target[:event_time], sm.add_constant(R_market[:event_time])).fit()

    resid = R_target - results.predict(sm.add_constant(R_market))

    t_stat = resid[event_time + 1]/resid[:event_time].std(ddof = 2)

    if abs(t_stat) > 1.96:
        detected += 1
    
    total += 1

print(detected/total)

/usr/local/python/3.14.2/lib/python3.14/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/usr/local/python/3.14.2/lib/python3.14/site-packages/numpy/_core/_methods.py:211: RuntimeWarning: divide by zero encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


0.04714142427281846
